# MPC spatial scene guide (waypoints → problem → solve)

Self-contained walkthrough aligned with `demo_dynamic_bicycle_rate_mpc_closed_loop_lap.py`:
workspace geometry → state costs → trajectory NLP → MPC closed loop.

**Conceptual map**

| Stage | Question answered |
| --- | --- |
| Workspace | Where is the track? Where are obstacles? |
| State fields | How do workspace queries depend on vehicle state $\mathbf{x}$? |
| `PlanningProblem` | What is the continuous optimal-control slice? |
| Transcription | How is the OCP turned into a finite NLP? |
| MPC | How is the NLP solved repeatedly in closed loop? |

**Further reading** (optional — same material as the theory blocks below)

| Topic | Where in course notes |
| --- | --- |
| Continuous OCP, direct collocation | Ch. 16 in Girard, Robotique: Modélisation, Analyse et Commande (UdeS course notes) |
| NLP, KKT, SQP | Ch. 28 in Girard, Robotique: Modélisation, Analyse et Commande (UdeS course notes) |
| Receding-horizon MPC | Ch. 13 in Girard, Robotique: Modélisation, Analyse et Commande (UdeS course notes) |

> Girard, A. *Robotique: Modélisation, Analyse et Commande*. Notes de cours, Université de Sherbrooke.

Companion script (spatial steps, no solve): `examples/scripts/mpc/demo_mpc_spatial_scene_guide.py`

Run from repo root with `pip install -e ".[jax]"`.

In [ ]:
# %matplotlib inline

# Colab setup (uncomment):
# !git clone https://github.com/alx87grd/minilink.git
# %cd minilink
# !pip install -q -e ".[jax]"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink.core.backends import configure_jax
from minilink.core.costs import QuadraticCost
from minilink.core.geometry import Sphere
from minilink.core.sets import BoxSet
from minilink.core.trajectory import Trajectory
from minilink.dynamics.catalog.vehicles.dynamic_bicycle import JaxDynamicBicycleRateInputs
from minilink.graphical.animation.primitives import HorizonPolyline, TrajectoryPolyline
from minilink.graphical.catalog import SceneHistory
from minilink.optimization.evaluators.compiler import compile_program_evaluator
from minilink.optimization.optimizer import Optimizer
from minilink.planning.initial_guess import default_initial_trajectory
from minilink.planning.mpc import (
    MPCDirectCollocationTranscription,
    MPCOptions,
    MPCPlanner,
)
from minilink.planning.problems import PlanningProblem
from minilink.planning.spatial.collision import bind, car_outline, point_probe
from minilink.planning.spatial.grid import pad_bounds, sample_field_costs
from minilink.planning.spatial.overlays import TrackCorridorOverlay
from minilink.planning.spatial.paths import from_waypoints
from minilink.planning.spatial.plotting import plot_cost_field_exports
from minilink.planning.spatial.scene import Scene
from minilink.planning.spatial.shaping import (
    inverse_barrier,
    quadratic_excess,
    quadratic_hinge,
)
from minilink.planning.spatial.track import ReferenceTrack
from minilink.planning.trajectory_optimization.direct_collocation import (
    DirectCollocationOptions,
    DirectCollocationTranscription,
)
from minilink.planning.trajectory_optimization.planner import (
    TrajectoryOptimizationOptions,
    TrajectoryOptimizationPlanner,
)

### Theory — continuous trajectory optimization

(Ch. 16 in Girard, Robotique: Modélisation, Analyse et Commande (UdeS course notes).)

A **trajectory optimization** problem seeks trajectories $\mathbf{x}(t), \mathbf{u}(t)$ that minimize

$$
\min_{\mathbf{x}(t),\,\mathbf{u}(t)} \;
J = \int_{t_0}^{t_f} g(\mathbf{x}, \mathbf{u})\, dt \;+\; h(\mathbf{x}(t_f))
$$

subject to

$$
\dot{\mathbf{x}}(t) - \mathbf{f}(\mathbf{x}, \mathbf{u}, t) = \mathbf{0}, \qquad
\mathbf{x}(t_0) = \mathbf{x}_{\mathrm{init}}, \qquad
\mathbf{x} \in \mathcal{X}_{\mathrm{ok}},\; \mathbf{u} \in \mathcal{U}_{\mathrm{ok}}.
$$

The dynamics $\mathbf{f}$ is provided by a minilink **`System`** (`sys`) — see **Step 2** for how `sys.f` flows into `PlanningProblem` and transcription.

**Two layers in this demo**

| Layer | Symbol | minilink objects |
| --- | --- | --- |
| Workspace | $\mathbf{p} \in \mathbb{R}^2$ | `ReferenceTrack`, `Scene`, heatmaps |
| State | $\mathbf{x} \in \mathbb{R}^n$ | `JaxDynamicBicycleRateInputs`; FK via `bind(sys, body)` |

Spatial terms enter $g$ after forward kinematics: probe positions $\mathbf{p}_k = \pi_k(\mathbf{x})$.

## Scenario parameters

Same compact CCW loop as `demo_dynamic_bicycle_rate_mpc_closed_loop_lap.py`:
24 m × 14 m rounded rectangle, three keepout spheres, soft spatial costs only at MPC runtime.


In [ ]:
TRACK_CENTER = (0.0, 0.0)
TRACK_WIDTH = 24.0
TRACK_HEIGHT = 14.0
TURN_RADIUS = 3.5
CORRIDOR_HALF_WIDTH = 2.0
UNROLL_LAPS = 3

OBSTACLE_RADIUS = 0.4
OBSTACLE_MARGIN = 0.05
OBSTACLE_CENTERS = (
    (4.0, -6.2),
    (-2.0, 5.0),
    (10.0, 1.5),
)

PATH_COST_WEIGHT = 40.0
CORRIDOR_COST_WEIGHT = 25.0
OBSTACLE_REPULSION_WEIGHT = 3.0
OBSTACLE_REPULSION_EPS = 0.08

U_TARGET = 20.0
VX0 = 2.5
TF_SIM = 16.0

MPC_HZ = 5.0
SIM_HZ = 200.0
MPC_HORIZON = 1.5
MPC_STEPS = 15
MPC_MAXITER = 120
MPC_FTOL = 1.0
MPC_DT = 1.0 / MPC_HZ
SIM_DT = 1.0 / SIM_HZ
SUBSTEPS = max(1, int(round(MPC_DT / SIM_DT)))

W_REAR_MAX = 90.0
DELTA_MAX = 0.55
W_REAR_DOT_MAX = 80.0
DELTA_DOT_MAX = 2.0
CAMERA_SCALE = 16.0
PLOT_MARGIN = 3.0
COST_FIELD_MARGIN = 4.0


def _quarter_arc_waypoints(cx, cy, radius, angle_start, n=12):
    angles = np.linspace(angle_start, angle_start + np.pi / 2, n)
    return np.column_stack([cx + radius * np.cos(angles), cy + radius * np.sin(angles)])


def rounded_rect_loop_waypoints(*, cx, cy, width, height, radius, n_arc=10):
    w2, h2, r = width / 2.0, height / 2.0, radius
    hx = w2 - r
    start = np.array([[cx - hx, cy - h2]])
    bottom = np.linspace([cx - hx, cy - h2], [cx + hx, cy - h2], 5)[1:]
    arc_br = _quarter_arc_waypoints(cx + hx, cy - h2 + r, r, -np.pi / 2, n_arc)[1:]
    right = np.linspace([cx + w2, cy - h2 + r], [cx + w2, cy + h2 - r], 4)[1:-1]
    arc_tr = _quarter_arc_waypoints(cx + hx, cy + h2 - r, r, 0.0, n_arc)[1:]
    top = np.linspace([cx + hx, cy + h2], [cx - hx, cy + h2], 5)[1:-1]
    arc_tl = _quarter_arc_waypoints(cx - hx, cy + h2 - r, r, np.pi / 2, n_arc)[1:]
    left = np.linspace([cx - w2, cy + h2 - r], [cx - w2, cy - h2 + r], 4)[1:-1]
    arc_bl = _quarter_arc_waypoints(cx - hx, cy - h2 + r, r, np.pi, n_arc)[1:]
    return np.vstack(
        [start, bottom, arc_br, right, arc_tr, top, arc_tl, left, arc_bl, start]
    )


def unroll_track(waypoints, n_laps=3):
    parts = [np.asarray(waypoints, dtype=float)]
    for _ in range(n_laps - 1):
        parts.append(parts[-1][1:])
    return np.vstack(parts)


LOOP_WAYPOINTS = rounded_rect_loop_waypoints(
    cx=TRACK_CENTER[0],
    cy=TRACK_CENTER[1],
    width=TRACK_WIDTH,
    height=TRACK_HEIGHT,
    radius=TURN_RADIUS,
)
REFERENCE_WAYPOINTS = unroll_track(LOOP_WAYPOINTS, n_laps=UNROLL_LAPS)
START_XY = LOOP_WAYPOINTS[0].copy()
PLOT_BOUNDS = (
    (
        TRACK_CENTER[0] - TRACK_WIDTH / 2 - PLOT_MARGIN,
        TRACK_CENTER[0] + TRACK_WIDTH / 2 + PLOT_MARGIN,
    ),
    (
        TRACK_CENTER[1] - TRACK_HEIGHT / 2 - PLOT_MARGIN,
        TRACK_CENTER[1] + TRACK_HEIGHT / 2 + PLOT_MARGIN,
    ),
)

## Step 1 — Workspace geometry

Let the reference centerline be a polyline $\mathcal{C}$ with arc-length parameter $s$, and let $w>0$ be the corridor half-width (`CORRIDOR_HALF_WIDTH`). For a workspace point $\mathbf{p} \in \mathbb{R}^2$:

$$
d(\mathbf{p}) = \min_{s} \|\mathbf{p} - \mathcal{C}(s)\|
\qquad\text{(distance to centerline)}
$$

$$
m(\mathbf{p}) = w - d(\mathbf{p})
\qquad\text{(corridor margin; $m>0$ inside the tube)}
$$

Hard obstacles $\mathcal{O} = \{O_j\}$ are shapes with signed-distance fields $\mathrm{sdf}_{O_j}$. The scene clearance is the **union SDF** (negative inside solids):

$$
c(\mathbf{p}) = \min_j \mathrm{sdf}_{O_j}(\mathbf{p})
\qquad\text{(positive in free space, zero on boundary, negative inside)}
$$

`from_waypoints` builds $\mathcal{C}$; `ReferenceTrack` stores $w$ and exposes $d(\cdot)$, $m(\cdot)$. `Scene` stores $\{O_j\}$ and exposes $c(\cdot)$.

- **`loop_track`** — one-lap $\mathcal{C}$ for maps, lap progress, animation.
- **`track`** — three-lap unroll of $\mathcal{C}$ for path/corridor **cost fields** (same as the lap demo script).

In [ ]:
loop_track = ReferenceTrack(
    from_waypoints(LOOP_WAYPOINTS), half_width=CORRIDOR_HALF_WIDTH
)
track = ReferenceTrack(
    from_waypoints(REFERENCE_WAYPOINTS), half_width=CORRIDOR_HALF_WIDTH
)
keepout_radius = OBSTACLE_RADIUS + OBSTACLE_MARGIN
scene = Scene(
    obstacles=tuple(Sphere(center, keepout_radius) for center in OBSTACLE_CENTERS)
)

sample_p = START_XY + np.array([2.0, 0.5])
print(f"lap length = {loop_track.path.total_length:.2f} m")
print(f"distance({sample_p}) = {loop_track.distance(sample_p):.3f}")
print(f"corridor_margin({sample_p}) = {loop_track.corridor_margin(sample_p):.3f}")
print(f"clearance({sample_p}) = {scene.clearance(sample_p):.3f}")

_, ax = scene.plot(show=False, bounds=PLOT_BOUNDS, show_density=False, title="")
loop_track.plot(show=False, ax=ax, bounds=PLOT_BOUNDS, title="")
ax.scatter([START_XY[0]], [START_XY[1]], c="C1", s=36, zorder=8, label="start")
ax.set_title("Workspace: track tube + keepouts")
ax.legend(loc="upper left", fontsize=8)
plt.show()

## Step 2 — Planner plant (`sys`)

### Role of `sys` in the optimization problem

In minilink, a **`System`** (here `JaxDynamicBicycleRateInputs`) is the **dynamics model** $\mathbf{f}$ in the OCP. It defines:

| `sys` attribute | Optimization meaning |
| --- | --- |
| `sys.n`, `sys.m` | dimensions of $\mathbf{x} \in \mathbb{R}^n$, $\mathbf{u} \in \mathbb{R}^m$ |
| `sys.f(x, u)` | right-hand side $\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x}, \mathbf{u})$ |
| state / input bounds | box parts of $\mathcal{X}_{\mathrm{ok}}$, $\mathcal{U}_{\mathrm{ok}}$ |
| forward kinematics (via `bind`) | maps $\mathbf{x}$ to workspace probes for spatial costs |

The **continuous** dynamics constraint is

$$
\dot{\mathbf{x}}(t) - \mathbf{f}(\mathbf{x}(t), \mathbf{u}(t)) = \mathbf{0}.
$$

**Where `sys` enters the pipeline**

1. **`PlanningProblem(sys=...)`** — attaches $\mathbf{f}$, state dimension, and default bounds to the continuous problem.
2. **`transcription.transcribe(problem)`** — reads `problem.sys.f` and turns the ODE into **defect equalities** $\mathbf{h}(\mathbf{z}) = \mathbf{0}$ on the collocation grid (direct collocation).
3. **`Optimizer.solve()`** — enforces those equalities together with $J(\mathbf{z})$ and inequality constraints from `X`.
4. **Closed-loop simulation (Step 11)** — a separate plant `sys_sim` integrates the **same** $\mathbf{f}$ with RK4 between MPC ticks; `sys_mpc` is the model **inside** the optimizer.

Spatial track/obstacle geometry does **not** live in `sys` — it enters through `cost` and optional `X` built from state fields. `sys` is only the **vehicle ODE + bounds + FK hook** for `bind`.

### This demo: rate-input bicycle

`JaxDynamicBicycleRateInputs`: wheel speed $\omega_r$ and steer $\delta$ are **states**; their rates $(\dot\omega_r, \dot\delta)$ are **inputs** (rate-MPC). We configure bounds before `bind` — even `point_probe` heatmaps need `sys.n`.


In [ ]:
configure_jax(enable_x64=True)

sys_mpc = JaxDynamicBicycleRateInputs()
sys_mpc.state.lower_bound[6] = 0.0
sys_mpc.state.upper_bound[6] = W_REAR_MAX
sys_mpc.state.lower_bound[7] = -DELTA_MAX
sys_mpc.state.upper_bound[7] = DELTA_MAX
sys_mpc.inputs["w_rear_dot"].lower_bound[0] = -W_REAR_DOT_MAX
sys_mpc.inputs["w_rear_dot"].upper_bound[0] = W_REAR_DOT_MAX
sys_mpc.inputs["delta_dot"].lower_bound[0] = -DELTA_DOT_MAX
sys_mpc.inputs["delta_dot"].upper_bound[0] = DELTA_DOT_MAX
print(f"n={sys_mpc.n}, m={sys_mpc.m}")

## Step 3 — Workspace cost heatmaps

Before attaching the vehicle body, rasterize the **workspace** part of the running cost at grid points $\mathbf{p}$. With shaping $s(\cdot)$ and weight $w$:

$$
\ell(\mathbf{p}) = w\, s\!\big(\phi(\mathbf{p})\big).
$$

This notebook uses three workspace scalars:

| Field | $\phi(\mathbf{p})$ | Shaping in demo |
| --- | --- | --- |
| Path | $d(\mathbf{p})$ | `quadratic_excess`: $s(d)=\max(d-d_0,0)^2$ |
| Corridor | $m(\mathbf{p})=w-d(\mathbf{p})$ | `quadratic_hinge`: $s(m)=\max(-m,0)^2$ |
| Obstacle | $c(\mathbf{p})$ | `inverse_barrier`: $s(c)=1/\max(c,\varepsilon)^2$ |

For a **point probe** at the grid location ($r=0$), $\phi(\mathbf{p})$ is evaluated directly. After Step 4, the same shapes become $\phi(\mathbf{x})=\min_k(\cdots)$ over body spheres.

`sample_field_costs` plots $\ell(\mathbf{p})$ on a 2-D grid — useful for tuning weights before running MPC.

In [ ]:
probe = bind(sys_mpc, point_probe())

path_cost_viz = track.distance_field(probe).as_cost(
    weight=PATH_COST_WEIGHT, shaping=quadratic_excess(threshold=0.1)
)
corridor_cost_viz = track.corridor_field(probe).as_cost(
    weight=CORRIDOR_COST_WEIGHT, shaping=quadratic_hinge(threshold=0.0)
)
obstacle_cost_viz = scene.clearance_field(probe).as_cost(
    weight=OBSTACLE_REPULSION_WEIGHT,
    shaping=inverse_barrier(epsilon=OBSTACLE_REPULSION_EPS),
)

cost_bounds = pad_bounds(PLOT_BOUNDS, COST_FIELD_MARGIN - PLOT_MARGIN)
_cost_kw = dict(bounds=cost_bounds, state_dim=sys_mpc.n, grid=(100, 100))
layers = {
    "path": sample_field_costs([path_cost_viz], **_cost_kw),
    "corridor": sample_field_costs([corridor_cost_viz], **_cost_kw),
    "obstacle": sample_field_costs([obstacle_cost_viz], **_cost_kw),
    "combined": sample_field_costs(
        [path_cost_viz, corridor_cost_viz, obstacle_cost_viz], **_cost_kw
    ),
}

plot_cost_field_exports(
    layers,
    track=loop_track,
    scene=scene,
    overlay_bounds=PLOT_BOUNDS,
    log_scale=True,
)

## Step 4 — Collision body: `bind(sys, geometry)`

Frameless geometry (discs in the body frame) is attached to the planner plant. Forward kinematics from `sys.tf(x)` place each probe in the world:

$$
\mathbf{p}_k(\mathbf{x}) = \mathbf{T}_{\mathrm{frame}(k)}(\mathbf{x})\, \mathbf{p}_{k,\mathrm{body}}, \qquad k = 1,\ldots,K.
$$

Each probe is a sphere $(\mathbf{p}_{k,\mathrm{body}}, r_k)$. `iter_probes(body, x)` yields world pairs $(\mathbf{p}_k(\mathbf{x}), r_k)$.

| Geometry | Probes | Role |
| --- | --- | --- |
| `point_probe()` | $K=1$, $r_1=0$ | heatmaps at grid $\mathbf{p}$ |
| `car_outline(...)` | $K>1$, $r_k>0$ | oriented footprint for MPC costs |

**State fields (Step 5)** always use the **minimum over probes** — the weakest margin across the footprint sets the scalar.

In [ ]:
body = bind(sys_mpc, car_outline(length=2.4, width=0.2, margin=0.05))

r_r = sys_mpc.params["r_r"]
w_rear_ref = U_TARGET / r_r
x_cruise = np.array([0.0, 0.0, 0.0, U_TARGET, 0.0, 0.0, w_rear_ref, 0.0])
ubar = np.zeros(sys_mpc.m)

s_start, _ = loop_track.path.project(START_XY)
tangent = loop_track.path.tangent(s_start)
theta0 = float(np.arctan2(tangent[1], tangent[0]))
if abs(np.cos(2.0 * theta0)) > 1.0 - 1e-9:
    theta0 += 1e-4
x0 = np.array([START_XY[0], START_XY[1], theta0, VX0, 0.0, 0.0, VX0 / r_r, 0.0])

### Probe vs body at the same pose

At a fixed state $\mathbf{x}$, compare one world point vs the full outline.

**Point probe** ($r=0$):

$$
c_{\mathrm{probe}}(\mathbf{x}) = c\!\big(\mathbf{p}_1(\mathbf{x})\big).
$$

**Car outline** ($K$ spheres):

$$
c_{\mathrm{body}}(\mathbf{x}) = \min_{k=1..K} \Big( c\!\big(\mathbf{p}_k(\mathbf{x})\big) - r_k \Big).
$$

Subtracting $r_k$ inflates each disc: the body is treated as a union of keepout spheres. The same pattern applies to path distance and corridor margin:

$$
d_{\mathrm{body}}(\mathbf{x}) = \min_k \big( d(\mathbf{p}_k(\mathbf{x})) - r_k \big), \qquad
m_{\mathrm{body}}(\mathbf{x}) = \min_k \big( w - d(\mathbf{p}_k(\mathbf{x})) - r_k \big).
$$

Heatmaps (Step 3) use the probe; MPC costs below use the body aggregates.

In [ ]:
_clear_body = scene.clearance_field(body).value(x0)
_clear_probe = scene.clearance_field(probe).value(x0)
print(f"clearance(body) @ x0 = {_clear_body:.3f}")
print(f"clearance(probe) @ x0 = {_clear_probe:.3f}")

## Step 5 — State fields: `value(x)`

A **state field** lifts a workspace function to state space by forward kinematics and a **probe reduction** (always `min` for clearance, path distance, and corridor margin in this demo).

Let $(\mathbf{p}_k(\mathbf{x}), r_k)_{k=1..K} = \texttt{iter\_probes(body, x)}$.

**Obstacle clearance field**

$$
\phi_{\mathrm{clear}}(\mathbf{x}) = \min_{k=1..K} \Big( c\!\big(\mathbf{p}_k(\mathbf{x})\big) - r_k \Big).
$$

Feasible when $\phi_{\mathrm{clear}}(\mathbf{x}) \geq 0$ (all inflated discs in free space).

**Path distance field**

$$
\phi_{\mathrm{path}}(\mathbf{x}) = \min_{k=1..K} \Big( d\!\big(\mathbf{p}_k(\mathbf{x})\big) - r_k \Big).
$$

**Corridor margin field**

$$
\phi_{\mathrm{cor}}(\mathbf{x}) = \min_{k=1..K} \Big( w - d\!\big(\mathbf{p}_k(\mathbf{x})\big) - r_k \Big).
$$

| Factory | Code | Reduction |
| --- | --- | --- |
| `scene.clearance_field(body)` | `ClearanceField` | $\min_k\big(c(\mathbf{p}_k)-r_k\big)$ |
| `track.distance_field(body)` | `PathDistanceField` | $\min_k\big(d(\mathbf{p}_k)-r_k\big)$ |
| `track.corridor_field(body)` | `CorridorMarginField` | $\min_k\big(w-d(\mathbf{p}_k)-r_k\big)$ |

Exports: $\phi(\mathbf{x})\geq 0$ → hard constraint (`as_constraint`); $w\,s(\phi(\mathbf{x}))$ → soft cost (`as_cost`).

In [ ]:
clearance_field = scene.clearance_field(body)
corridor_field = track.corridor_field(body)
distance_field = track.distance_field(body)

for name, field in [
    ("clearance", clearance_field),
    ("corridor", corridor_field),
    ("path distance", distance_field),
]:
    print(f"{name:14s} value(x0) = {float(field.value(x0)):.4f}")

## Step 6 — Hard set: `field.as_constraint()`

Each scalar field defines a feasible set

$$
\mathcal{X}_\phi = \{\mathbf{x} : \phi(\mathbf{x}) \geq \phi_{\min}\}.
$$

`FieldSet(lower=0)` means $\phi(\mathbf{x}) \geq 0$. For collocation node $\mathbf{x}_k$, the **inequality margin** exported to the NLP is

$$
g_{\phi,k}(\mathbf{z}) = \phi(\mathbf{x}_k) - \phi_{\min} \geq 0.
$$

This notebook assembles

$$
\mathcal{X}_{\mathrm{ok}}
= \mathcal{X}_{\mathrm{box}}
\cap \mathcal{X}_{\mathrm{clear}}
\cap \mathcal{X}_{\mathrm{cor}}
$$

with

$$
\mathcal{X}_{\mathrm{box}} = \{\mathbf{x} : \mathbf{x}_{\min} \leq \mathbf{x} \leq \mathbf{x}_{\max}\}, \quad
\mathcal{X}_{\mathrm{clear}} = \{\mathbf{x} : \phi_{\mathrm{clear}}(\mathbf{x}) \geq 0\}, \quad
\mathcal{X}_{\mathrm{cor}} = \{\mathbf{x} : \phi_{\mathrm{cor}}(\mathbf{x}) \geq 0\}.
$$

At each node $k$, all margins from $\mathcal{X}_{\mathrm{ok}}$ stack into $\mathbf{g}(\mathbf{z})\geq \mathbf{0}$.

We build $\mathcal{X}$ here to show the pattern. **Steps 8–11 omit `X=X`** (same as the lap demo): on this short horizon with the full car outline, hard clearance + corridor constraints make the single-horizon NLP infeasible for SLSQP — the optimizer returns $x_0$ and Step 9d shows no motion. Spatial keep-out enters through soft barrier/hinge costs in $J$ instead.

### Hard vs soft — when to use which

| Export | NLP form | Meaning |
| --- | --- | --- |
| `as_constraint(lower=0)` | $\phi(\mathbf{x}_k)\geq 0$ at every node | **must** stay feasible (tube, free space) |
| `as_cost(shaping=...)` | add $w\,s(\phi(\mathbf{x}_k))$ to $J$ | **penalty** — trade-offs allowed |

Hard constraints shrink the feasible set; soft costs reshape $J$ while leaving violations possible if other terms dominate.

In [ ]:
X = (
    BoxSet.from_system_state(sys_mpc)
    & clearance_field.as_constraint(lower=0.0)
    & corridor_field.as_constraint(lower=0.0)
)
print("X assembled: bounds ∩ obstacle clearance ≥ 0 ∩ corridor margin ≥ 0")

## Step 7 — Soft cost: `field.as_cost(shaping=...)`

Each spatial export is a **FieldCost**:

$$
g_{\mathrm{field}}(\mathbf{x}, \mathbf{u}) = w\, s\!\big(\phi(\mathbf{x})\big), \qquad h_{\mathrm{field}}(\mathbf{x}) = 0.
$$

The full running cost in this demo is

$$
g(\mathbf{x}, \mathbf{u}) =
(\mathbf{x}-\bar{\mathbf{x}})^\top Q (\mathbf{x}-\bar{\mathbf{x}})
+ (\mathbf{u}-\bar{\mathbf{u}})^\top R (\mathbf{u}-\bar{\mathbf{u}})
+ w_p\, s_{\mathrm{excess}}\!\big(\phi_{\mathrm{path}}(\mathbf{x})\big)
+ w_c\, s_{\mathrm{hinge}}\!\big(\phi_{\mathrm{cor}}(\mathbf{x})\big)
+ w_o\, s_{\mathrm{barrier}}\!\big(\phi_{\mathrm{clear}}(\mathbf{x})\big).
$$

With the explicit workspace-to-state maps from Step 5:

$$
s_{\mathrm{excess}}(\phi) = \max(\phi - d_0, 0)^2, \quad
s_{\mathrm{hinge}}(\phi) = \max(-\phi, 0)^2, \quad
s_{\mathrm{barrier}}(\phi) = \frac{1}{\max(\phi, \varepsilon)^2}.
$$

Weights $(w_p, w_c, w_o) =$ (`PATH_COST_WEIGHT`, `CORRIDOR_COST_WEIGHT`, `OBSTACLE_REPULSION_WEIGHT`) match `demo_dynamic_bicycle_rate_mpc_closed_loop_lap.py`. Compose with `+` on `CostFunction`.

In [ ]:
_v = np.linspace(-0.5, 3.0, 200)
fig, ax = plt.subplots(figsize=(7.0, 3.5))
ax.plot(_v, [float(quadratic_excess(0.1)(vi)) for vi in _v], label="quadratic_excess (path)")
ax.plot(_v, [float(quadratic_hinge(0.0)(vi)) for vi in _v], label="quadratic_hinge (corridor)")
ax.plot(_v, [float(inverse_barrier(OBSTACLE_REPULSION_EPS)(vi)) for vi in _v], label="inverse_barrier (obstacle)")
ax.set_xlabel("raw field value v")
ax.set_ylabel("shaping(v)")
ax.set_title("Shaping functions (before weighting)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
stability_cost = QuadraticCost.from_system(
    sys_mpc,
    Q=np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0]),
    R=np.diag([1.0, 22.0]),
    S=np.diag([0.0, 0.0, 0.0, 0.15, 4.0, 6.0, 0.1, 80.0]),
    xbar=x_cruise,
    ubar=ubar,
)
path_cost = distance_field.as_cost(
    weight=PATH_COST_WEIGHT, shaping=quadratic_excess(threshold=0.1)
)
corridor_cost = corridor_field.as_cost(
    weight=CORRIDOR_COST_WEIGHT, shaping=quadratic_hinge(threshold=0.0)
)
obstacle_cost = clearance_field.as_cost(
    weight=OBSTACLE_REPULSION_WEIGHT,
    shaping=inverse_barrier(epsilon=OBSTACLE_REPULSION_EPS),
)
cost = stability_cost + path_cost + corridor_cost + obstacle_cost

## Step 8 — `PlanningProblem`

For one horizon $[t_0, t_0+T]$, the continuous slice is

$$
\min_{\mathbf{x}(\cdot), \mathbf{u}(\cdot)} \;
\int_{t_0}^{t_0+T} g(\mathbf{x}, \mathbf{u})\, d\tau
\quad \text{s.t.} \quad
\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x}, \mathbf{u}),\;
\mathbf{x}(t_0) = \mathbf{x}_{\mathrm{meas}},\;
\mathbf{x}(t) \in \mathcal{X}_{\mathrm{ok}},\;
\mathbf{u}(t) \in \mathcal{U}_{\mathrm{ok}}.
$$

`PlanningProblem` packages this:

| Field | Math object |
| --- | --- |
| `sys` | $\mathbf{f}$, $(n,m)$, default input/state boxes |
| `x_start` | $\mathbf{x}(t_0)$ |
| `X` | optional hard $\mathcal{X}_{\mathrm{ok}}$ (Step 6) |
| `cost` | $g$ (quadratic + spatial soft terms) |

**Single-horizon solve (Steps 9–10) and MPC (Step 11)** use **soft spatial costs only** — same as the lap demo:

```python
problem = PlanningProblem(sys=sys_mpc, x_start=x0, cost=cost)
```

Hard $\mathcal{X}_{\mathrm{clear}} \cap \mathcal{X}_{\mathrm{cor}}$ from Step 6 is pedagogical: on this short horizon with the full car outline, passing `X=X` makes the NLP **infeasible** for SLSQP (the optimizer returns the initial point and the Step 9d plot shows no motion). Spatial keep-out is enforced via soft barrier/hinge costs instead.

In [ ]:
problem = PlanningProblem(sys=sys_mpc, x_start=x0, cost=cost)
print(problem)
print("(soft spatial costs only — same pattern as MPC; hard X from Step 6 omitted for feasible solve)")

## Step 9 — Transcription: continuous → finite NLP

(Ch. 16 in Girard, Robotique: Modélisation, Analyse et Commande (UdeS course notes).)

Discretize $t_0,\ldots,t_N$ with $\Delta t = T/N$. **Direct collocation** treats $\mathbf{x}_0,\ldots,\mathbf{x}_N$ and $\mathbf{u}_0,\ldots,\mathbf{u}_{N-1}$ as decision variables.

**Stage cost** (trapezoidal quadrature):

$$
J(\mathbf{z}) = \sum_{k=0}^{N-1} g(\mathbf{x}_k, \mathbf{u}_k)\,\Delta t + h(\mathbf{x}_N).
$$

**Dynamics defects** (equality):

$$
\mathbf{h}_k(\mathbf{z}) = \mathbf{x}_{k+1} - \mathbf{f}_d(\mathbf{x}_k, \mathbf{u}_k) = \mathbf{0}, \qquad
\mathbf{x}_0 - \mathbf{x}_{\mathrm{init}} = \mathbf{0}.
$$

**Path / corridor / clearance** at node $k$ (if hard `X` is used):

$$
g_{\phi,k}(\mathbf{z}) = \phi(\mathbf{x}_k) - \phi_{\min} \geq 0.
$$

Pack

$$
\mathbf{z} = \big[\mathbf{x}_0^\top, \ldots, \mathbf{x}_{N-1}^\top,\; \mathbf{u}_0^\top, \ldots, \mathbf{u}_{N-1}^\top\big]^\top.
$$

NLP:

$$
\min_{\mathbf{z}} J(\mathbf{z}) \quad \text{s.t.} \quad \mathbf{h}(\mathbf{z}) = \mathbf{0},\; \mathbf{g}(\mathbf{z}) \geq \mathbf{0},\; \mathbf{z}_{\min} \leq \mathbf{z} \leq \mathbf{z}_{\max}.
$$

Chain: `PlanningProblem` → `transcribe()` → `MathematicalProgram` → `Optimizer.solve()` → `reconstruct_result()` → `Trajectory`.

In [ ]:
COMPILE_BACKEND = "jax"

transcription = DirectCollocationTranscription(
    DirectCollocationOptions(tf=MPC_HORIZON, n_steps=MPC_STEPS)
)
trajopt_options = TrajectoryOptimizationOptions(
    compile_backend=COMPILE_BACKEND,
    optimizer_method="scipy_slsqp",
    solve_disp=False,
    record_solve_time=True,
    optimizer_options={"maxiter": MPC_MAXITER, "ftol": MPC_FTOL},
)
n, m, n_steps = sys_mpc.n, sys_mpc.m, MPC_STEPS
print(f"horizon={MPC_HORIZON}s, N={n_steps} nodes")
print(f"decision dim n_z = (n + m) * N = ({n} + {m}) * {n_steps} = {(n + m) * n_steps}")

### Step 9a — `transcribe(problem)` → `MathematicalProgram`

The transcription evaluates `problem.cost.g(x,u)` at each node $\mathbf{x}_k$ and builds:

$$
J(\mathbf{z}) = \sum_k g(\mathbf{x}_k, \mathbf{u}_k)\,\Delta t, \qquad
\mathbf{h}(\mathbf{z}) = \big[\mathbf{h}_{\mathrm{dyn}};\; \mathbf{h}_{\mathrm{bc}}\big], \qquad
\mathbf{g}(\mathbf{z}) = \big[g_{\mathrm{state},k}\big].
$$

With soft-only spatial costs, $\mathbf{g}$ comes from **state/input boxes** only (no hard clearance/corridor rows). Spatial keep-out enters through $J$ via $\phi_{\mathrm{clear}}$, $\phi_{\mathrm{cor}}$, $\phi_{\mathrm{path}}$.

- `n_h` — dynamics defects + pinned $\mathbf{x}_0$
- `n_g` — box slack margins on $\mathbf{x}_k$, $\mathbf{u}_k$

In [ ]:
program = transcription.transcribe(problem, compile_backend=COMPILE_BACKEND)
evaluator = compile_program_evaluator(program, sample_z=np.zeros(program.n_z))
print("MathematicalProgram")
print(f"  n_z   = {program.n_z}")
print(f"  n_h   = {evaluator.n_h}  (equalities: dynamics defects + boundaries)")
print(f"  n_g   = {evaluator.n_g}  (inequalities: path/corridor/obstacle margins)")
print(f"  backend = {program.backend!r}")

### Step 9b — Initial guess on the collocation grid

`initial_guess_time_grid` returns `t = linspace(0, tf, N)`. `default_initial_trajectory` holds `x_start` when no goal is set. `pack_initial_guess` flattens `(x, u)` into `z0`.


In [ ]:
t_grid = transcription.initial_guess_time_grid(problem)
guess = default_initial_trajectory(problem, t_grid)
z0 = transcription.pack_initial_guess(problem, guess)

evaluator = compile_program_evaluator(program, sample_z=z0)
J0 = float(evaluator.objective(z0))
max_eq, min_ineq, max_bound = evaluator.constraint_violations(z0)
min_ineq_s = "n/a (n_g=0)" if min_ineq is None else f"{min_ineq:.2e}"
print(f"t_grid: {t_grid[0]:.2f} … {t_grid[-1]:.2f} s  ({t_grid.size} nodes)")
print(f"z0 shape = {z0.shape}")
print(f"J(z0) = {J0:.4f}")
print(
    f"violations @ z0: max_eq={max_eq:.2e}, min_ineq={min_ineq_s}, max_bound={max_bound:.2e}"
)

**Reading constraint violations**

- `max_eq` — largest |equality residual|; should be ≈ 0 when dynamics and pinned `x_start` are satisfied.
- `min_ineq` — smallest inequality margin `g(z)`; `None` when `n_g=0` (this notebook: soft spatial costs only). Otherwise should be ≥ 0 for feasible hard margins.
- `max_bound` — box-bound slack violation on `z`.


In [ ]:
x0_unpack, u0_unpack = transcription.unpack(z0, problem)
print("x[:, 0] from z0:", np.round(x0_unpack[:, 0], 3))
print("x_start:       ", np.round(problem.x_start, 3))
print("match:", np.allclose(x0_unpack[:, 0], problem.x_start, atol=1e-9))

### Step 9c — `Optimizer(program, z0).solve()` → `OptimizationResult`

(Ch. 28 in Girard, Robotique: Modélisation, Analyse et Commande (UdeS course notes).)

**Theory.** The solver iterates on the NLP. SLSQP uses **sequential quadratic programming (SQP)**: at each iterate $\mathbf{z}_k$, linearize $\mathbf{h}, \mathbf{g}$ and build a local QP subproblem to obtain a search direction.

**KKT conditions** at a candidate $\mathbf{z}^\star$ (first-order optimality for constrained problems):

$$
\nabla_{\mathbf{z}} \mathcal{L} = \mathbf{0}, \quad
\mathbf{h}(\mathbf{z}^\star) = \mathbf{0}, \quad
\mathbf{g}(\mathbf{z}^\star) \leq \mathbf{0}, \quad
\boldsymbol{\lambda} \geq \mathbf{0}, \quad
\boldsymbol{\lambda}^\top \mathbf{g}(\mathbf{z}^\star) = 0.
$$

with Lagrangian $\mathcal{L} = J + \boldsymbol{\nu}^\top \mathbf{h} + \boldsymbol{\lambda}^\top \mathbf{g}$.

Non-convex bicycle dynamics → **local** optimum only; warm-start and a good initial guess matter.

In [ ]:
optimizer = Optimizer(
    program,
    z0=z0,
    method=trajopt_options.optimizer_method,
    use_hessian=trajopt_options.use_hessian,
    options=dict(trajopt_options.optimizer_options),
)
manual_result = optimizer.solve(record_solve_time=True)

max_eq, min_ineq, max_bound = optimizer.program_evaluator.constraint_violations(
    manual_result.z
)
print(f"success = {manual_result.success}")
print(f"message = {manual_result.message}")
print(f"J* = {manual_result.cost:.4f}")
print(f"solve_time = {manual_result.solve_time_s:.3f} s")
min_ineq_s = "n/a (n_g=0)" if min_ineq is None else f"{min_ineq:.2e}"
print(
    f"violations @ z*: max_eq={max_eq:.2e}, min_ineq={min_ineq_s}, max_bound={max_bound:.2e}"
)

### Step 9d — `reconstruct_result` → `Trajectory`

Unpack `z*` back into `(x, u)` matrices on the collocation grid, then plot the standard minilink time signals via `sys.plot_trajectory()`.

In [ ]:
manual_traj = transcription.reconstruct_result(
    manual_result,
    problem=problem,
    compile_backend=COMPILE_BACKEND,
)
print(f"trajectory: {manual_traj.n_samples} samples, tf={manual_traj.t[-1]:.2f} s")
print(f"x(0)  xy = ({manual_traj.x[0,0]:.2f}, {manual_traj.x[1,0]:.2f})")
print(f"x(tf) xy = ({manual_traj.x[0,-1]:.2f}, {manual_traj.x[1,-1]:.2f})")
if not manual_result.success:
    print(f"warning: solve failed ({manual_result.message!r}) — check problem feasibility")

sys_mpc.traj = manual_traj
sys_mpc.plot_trajectory(signals=("x", "u"))

## Step 10 — `TrajectoryOptimizationPlanner` wraps the same chain

`compute_solution()` orchestrates Step 9a–9d on the discrete collocation problem above. The planner caches `last_program`, `last_optimizer`, and `last_optimization_result`.

Manual vs wrapper should agree on $J^\star$ and $(x,y)$ trajectory when given the same `problem`, `z0`, and solver options.


In [ ]:
planner = TrajectoryOptimizationPlanner(
    problem,
    transcription=transcription,
    options=trajopt_options,
)
planner_traj = planner.compute_solution(initial_guess=guess)
wrapper_result = planner.last_optimization_result

print("Planner wrapper vs manual pipeline:")
print(f"  n_z match: {planner.last_program.n_z == program.n_z}")
print(f"  J* manual  = {manual_result.cost:.6f}")
print(f"  J* planner = {wrapper_result.cost:.6f}")
print(f"  xy close: {np.allclose(planner_traj.x[0:2], manual_traj.x[0:2], atol=1e-3)}")

---

## Step 11 — MPC closed loop (`MPCPlanner`)

(Ch. 13 in Girard, Robotique: Modélisation, Analyse et Commande (UdeS course notes).)

**Theory — receding horizon.** At each time $t$, solve a **finite-horizon** trajectory optimization from the **measured** state; apply only the first control; shift the window:

$$
\mathbf{u}(t) = \mathbf{u}_0^\star \quad \text{where} \quad
\{\mathbf{x}^\star_{0:N}, \mathbf{u}^\star_{0:N-1}\}
= \arg\min \sum_{k=0}^{N-1} g(\mathbf{x}_k, \mathbf{u}_k)\,\Delta t
\;\; \text{s.t.}\;\; \mathbf{x}_0 = \mathbf{x}(t),\; \text{dynamics + constraints}.
$$

Then simulate $\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x}, \mathbf{u}(t))$ until $t \leftarrow t + \Delta t_{\mathrm{MPC}}$ and repeat. This is **implicit feedback** $\mathbf{u} = \pi(\mathbf{x})$ without storing a global policy.

**This notebook**

- Same task/tuning as `demo_dynamic_bicycle_rate_mpc_closed_loop_lap.py`
- **Compile-once** `MPCPlanner` (parametric $\mathbf{x}_0$) instead of rebuilding the NLP each tick
- Soft spatial costs only; warm-start from the time-shifted previous plan

In [ ]:
sys_sim = JaxDynamicBicycleRateInputs()
sys_sim.params["mass"] = 1.03 * sys_mpc.params["mass"]
sys_sim.params["inertia"] = 1.02 * sys_mpc.params["inertia"]
for sys in (sys_mpc, sys_sim):
    sys.state.lower_bound[6] = 0.0
    sys.state.upper_bound[6] = W_REAR_MAX
    sys.state.lower_bound[7] = -DELTA_MAX
    sys.state.upper_bound[7] = DELTA_MAX
    sys.inputs["w_rear_dot"].lower_bound[0] = -W_REAR_DOT_MAX
    sys.inputs["w_rear_dot"].upper_bound[0] = W_REAR_DOT_MAX
    sys.inputs["delta_dot"].lower_bound[0] = -DELTA_DOT_MAX
    sys.inputs["delta_dot"].upper_bound[0] = DELTA_DOT_MAX

sim_evaluator = sys_sim.compile(backend="jax", verbose=False)
template_problem = PlanningProblem(sys=sys_mpc, x_start=x0, cost=cost)
mpc_planner = MPCPlanner(
    template_problem,
    transcription=MPCDirectCollocationTranscription(
        DirectCollocationOptions(tf=MPC_HORIZON, n_steps=MPC_STEPS)
    ),
    options=MPCOptions(
        compile_backend="jax",
        optimizer_method="scipy_slsqp",
        record_solve_time=True,
        optimizer_options={"maxiter": MPC_MAXITER, "ftol": MPC_FTOL},
    ),
)
lap_length = loop_track.path.total_length
print(f"MPC compile={mpc_planner.compile_time_s:.3f}s (once)")
print(
    f"lap length={lap_length:.1f} m, u_target={U_TARGET} m/s, "
    f"tf_sim={TF_SIM:.1f} s, {len(OBSTACLE_CENTERS)} obstacles"
)

In [ ]:
t_hist, x_hist, u_hist = [0.0], [x0.copy()], [np.zeros(sys_sim.m)]
mpc_plans, progress_hist = [], [s_start]
x, t, u_hold = x0.copy(), 0.0, np.zeros(sys_sim.m)
prev_plan, next_mpc_t = None, 0.0

while t < TF_SIM - 1e-12:
    if t >= next_mpc_t - 1e-12:
        guess_mpc = None
        if prev_plan is not None and prev_plan.n_samples >= 3:
            t_shift = prev_plan.t + MPC_DT
            mask = t_shift <= MPC_HORIZON + 1e-9
            if np.count_nonzero(mask) >= 3:
                x_guess = prev_plan.x[:, mask].copy()
                x_guess[:, 0] = x
                guess_mpc = Trajectory(
                    t=t_shift[mask] - t, x=x_guess, u=prev_plan.u[:, mask]
                )
        else:
            guess_mpc = default_initial_trajectory(
                template_problem,
                mpc_planner.transcription.initial_guess_time_grid(template_problem),
            )

        plan = mpc_planner.step(x, initial_guess=guess_mpc)
        res = mpc_planner.last_optimization_result
        print(
            f"MPC @ t={t:.2f}s  success={res.success}  "
            f"solve={res.solve_time_s:.3f}s  step={mpc_planner.last_step_time_s:.3f}s"
        )
        prev_plan = plan
        u_hold = plan.u[:, 0].copy()
        mpc_plans.append(
            (t, Trajectory(t=plan.t + t, x=plan.x.copy(), u=plan.u.copy()))
        )
        next_mpc_t += MPC_DT

    for _ in range(SUBSTEPS):
        if t >= TF_SIM:
            break
        x = sim_evaluator.rk4_step(x, u_hold, t, SIM_DT)
        t += SIM_DT
        t_hist.append(t)
        x_hist.append(x.copy())
        u_hist.append(u_hold.copy())
        progress_hist.append(loop_track.path.project(x[0:2])[0])

traj = Trajectory(t=np.asarray(t_hist), x=np.asarray(x_hist).T, u=np.asarray(u_hist).T)
progress = np.asarray(progress_hist)
progress_delta = np.diff(progress, prepend=progress[0])
progress_delta = np.where(
    progress_delta < -0.5 * lap_length, progress_delta + lap_length, progress_delta
)
laps_completed = np.cumsum(np.maximum(progress_delta, 0.0))[-1] / lap_length
clearances = [
    np.hypot(traj.x[0, :] - cx, traj.x[1, :] - cy) - OBSTACLE_RADIUS
    for cx, cy in OBSTACLE_CENTERS
]
print(
    f"done: laps~={laps_completed:.2f}, mean vx={float(np.mean(traj.x[3, :])):.2f} m/s, "
    f"max vx={float(np.max(traj.x[3, :])):.2f} m/s, "
    f"min obstacle clearance={float(np.min(clearances)):.2f} m"
)

## Results

Executed path on the scene, then state/input histories and animation.


In [ ]:
fig, ax = loop_track.plot(
    show=False, bounds=PLOT_BOUNDS, title="Closed loop track + MPC executed path"
)
scene.plot(show=False, ax=ax, bounds=PLOT_BOUNDS, show_density=False, title=None)
ax.plot(traj.x[0, :], traj.x[1, :], color="tab:blue", linewidth=1.8, label="executed")
ax.scatter([START_XY[0]], [START_XY[1]], c="C1", s=36, zorder=8, label="start/finish")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
plt.show()

sys_sim.params = dict(sys_sim.params)
sys_sim.camera_scale = CAMERA_SCALE
sys_sim.traj = traj
sys_sim.plot_trajectory(signals=("x", "u"))

In [ ]:
history = SceneHistory(
    trail=TrajectoryPolyline(traj, window="prefix", color="b", style="--", linewidth=1.0),
    horizon=HorizonPolyline(mpc_plans, color="tab:orange", linewidth=2.0, style="--"),
)
sys_sim.animate(
    traj,
    overlays=[
        TrackCorridorOverlay(loop_track),
        scene.as_visualizer(color="tab:red", opacity=0.45),
        history,
    ],
)

---

## Recap and extensions

**Pipeline (one line):** workspace geometry → state fields → `PlanningProblem` → direct collocation NLP → `MPCPlanner` closed loop.

**Theory map → course notes**

| Notebook block | Ch. in Girard, Robotique: Modélisation, Analyse et Commande (UdeS course notes) |
| --- | --- |
| Continuous OCP (intro, Step 9) | 16 — *Optimisation de trajectoires* |
| KKT / SQP (Step 9c) | 28 — *Optimisation* |
| Receding horizon (Step 11) | 13 — *Introduction à la commande optimale* |

> Girard, A. *Robotique: Modélisation, Analyse et Commande*. Notes de cours, Université de Sherbrooke.

**Next in minilink:** `demo_dynamic_bicycle_rate_mpc_closed_loop_lap.py`, `demo_dynamic_bicycle_rate_mpc_fast_stadium_lap.py`, `DESIGN.md` (planning contracts).